In [ ]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(r"C:\Users\sjs93\Downloads\lfp_kinematics_habit_dishabit.csv")


In [ ]:
# --------------------------------------------------
# UM hierarchy ranks used for relative-rank analysis
#
# These UM (urine marking) ranks were taken from:
# “Just the results from 8/30 - done during habit dishabit”
#
# We specifically used the UM hierarchy data collected
# during the Habituation/Dishabituation phase because:
# - it was temporally aligned with the recordings
# - HCO rankings were incomplete for Cage 4
# - it provided complete hierarchy information across cages
#
# Lower numerical rank = more dominant
# --------------------------------------------------

um_rank_map = {
    "1.1": 2,
    "1.2": 1,
    "1.3": 3,

    "2.1": 1,
    "2.2": 2,
    "2.3": 4,
    "2.4": 2,

    "3.1": 2,
    "3.2": 3,
    "3.3": 1,

    "4.1": 3,
    "4.3": 1,
    "4.4": 1
}

## Habituation/Dishabituation Partner Schedule

The `schedule` dictionary maps each subject mouse to:
- the repeating cagemate (`A`)
- the novel cagemate (`B`)

used during the Habituation/Dishabituation paradigm in
Phase 1 of Meghan Cum’s SocialMemoryEphys Pilot 2 experiment.

These mappings were manually generated from the experimental
trial schedule spreadsheet: https://uflorida.sharepoint.com/:x:/r/teams/Padilla-CoreanoLab/Shared%20Documents/General/Data/SocialMemoryEphysPilot2/social_mem_ephys_pilot2_schedule%201.xlsx?d=w67ee7b5b2cdf4f59835b9b96c6b1313b&csf=1&web=1&e=Eryf8m

Experimental structure:
- Exposures 1–4 (`A`) = repeating social partner
- Exposure 5 (`B`) = novel social partner

This mapping was used to:
1. identify the actual partner mouse for each sniff event
2. append partner-specific hierarchy ranks
3. compute subject-relative hierarchy relationships

Example:
- Subject `1.1`
    - repeating partner (`A`) = `1.2`
    - novel partner (`B`) = `1.3`

Relative rank was then calculated using the UM hierarchy
ranks collected during the Habituation/Dishabituation phase.

In [ ]:
schedule = {

    "1.1": {"A": "1.2", "B": "1.3"},
    "1.2": {"A": "1.3", "B": "1.1"},
    "1.3": {"A": "1.1", "B": "1.2"},

    "2.1": {"A": "2.4", "B": "2.3"},
    "2.2": {"A": "2.3", "B": "2.4"},
    "2.3": {"A": "2.2", "B": "2.1"},
    "2.4": {"A": "2.1", "B": "2.2"},

    "3.1": {"A": "3.2", "B": "3.3"},
    "3.2": {"A": "3.3", "B": "3.1"},
    "3.3": {"A": "3.1", "B": "3.2"},

    "4.1": {"A": "4.3", "B": "4.4"},
    "4.4": {"A": "4.1", "B": "4.3"},
}

In [ ]:
# Convert subject and partner columns to strings
#
# This ensures that:
# - subject IDs (e.g., "1.1")
# - partner labels (e.g., "A" and "B")
#
# are treated as text rather than numeric values.
#
# This step is important because the downstream:
# - um_rank_map dictionary
# - schedule dictionary
#
# use STRING keys for lookup/mapping operations.
#
# Without converting to strings, pandas may interpret:
# - subject IDs as floats
# - partner labels inconsistently
#
# which can cause dictionary mapping failures and
# produce missing (NaN) values during:
# - partner identity assignment
# - hierarchy-rank mapping
# - relative-rank calculations

df["subject"] = df["subject"].astype(str)
df["partner"] = df["partner"].astype(str)

In [ ]:
# --------------------------------------------------
# Map experimental partner labels ("A" / "B")
# to the actual social partner mouse ID
#
# Each row represents a neural spectral measurement
# (e.g., coherence/power/granger) computed from
# social-investigation behavioral epochs.
#
# The original dataframe stores:
# - the subject mouse
# - whether the social interaction partner was:
#
#     "A" = repeating cagemate
#     "B" = novel cagemate
#
# This function uses the experimentally-defined
# Habituation/Dishabituation schedule to identify
# the actual cagemate mouse corresponding to each
# row.
#
# Example:
# subject = "1.1"
# partner label = "A"
#
# -> repeating cagemate = "1.2"
#
# The returned value becomes the:
#     partner_mouse
#
# column used for downstream:
# - hierarchy-rank mapping
# - relative-rank analysis
# - GLMs
# --------------------------------------------------

def get_partner_mouse(row):

    # extract subject mouse ID
    subj = row["subject"]

    # extract partner label
    # ("A" = repeating cagemate,
    #  "B" = novel cagemate)
    label = row["partner"]

    # verify:
    # - subject exists in schedule dictionary
    # - partner label is valid
    if subj in schedule and label in ["A", "B"]:

        # return actual cagemate mouse ID
        return schedule[subj][label]

    # return NaN if mapping fails
    return np.nan

In [ ]:
# Apply the partner-mapping function row-by-row
#
# For each neural-analysis observation:
# - read the subject mouse ID
# - read whether the interaction occurred with:
#     "A" = repeating cagemate
#     "B" = novel cagemate
#
# The get_partner_mouse() function then uses the
# Habituation/Dishabituation schedule dictionary
# to identify the actual cagemate mouse involved
# in that observation.
#
# The resulting partner mouse ID is stored in:
#     df["partner_mouse"]
#
# Example:
# subject = "1.1"
# partner label = "A"
#
# -> repeating cagemate = "1.2"
#
# axis=1 tells pandas to apply the function
# across rows.
#
# Each row represents a neural spectral observation
# (e.g., coherence/power/granger) computed from
# social-investigation behavioral epochs during the
# Habituation/Dishabituation paradigm.

df["partner_mouse"] = df.apply(
    get_partner_mouse,
    axis=1
)

In [ ]:
# --------------------------------------------------
# Append UM hierarchy ranks to the dataframe
#
# The um_rank_map dictionary contains the UM
# (urine marking) hierarchy rank assigned to
# each mouse during the Habituation/Dishabituation
# phase of the experiment.
#
# Lower numerical rank = more dominant
#
# Example:
# "1.2" -> rank 1 (more dominant)
# "1.3" -> rank 3 (less dominant)
#
# These hierarchy values are appended for:
# - the subject mouse
# - the interacting cagemate
#
# allowing downstream comparison of:
# - subject hierarchy position
# - repeating/novel cagemate hierarchy position
#
# which is later used to compute:
# - subject-relative hierarchy relationships
# - GLMs examining hierarchy-related effects on
#   neural spectral measurements
# --------------------------------------------------

# Map UM hierarchy rank for the subject mouse
#
# Example:
# subject = "1.1"
# -> subject_um_rank = 2

df["subject_um_rank"] = (
    df["subject"]
    .map(um_rank_map)
)

# Map UM hierarchy rank for the interacting cagemate
#
# Example:
# partner_mouse = "1.2"
# -> partner_um_rank = 1

df["partner_um_rank"] = (
    df["partner_mouse"]
    .map(um_rank_map)
)

In [ ]:
# --------------------------------------------------
# Compute the hierarchy relationship between:
# - the subject mouse
# - the interacting cagemate
#
# This function compares the UM hierarchy ranks
# assigned to:
# - the subject mouse
# - the repeating or novel cagemate
#
# to determine whether the interacting cagemate is:
#
# - more dominant than the subject
# - less dominant than the subject
# - equal hierarchy rank
#
# IMPORTANT:
# Lower numerical rank = more dominant
#
# Example:
#
# subject_um_rank = 2
# partner_um_rank = 1
#
# Since:
#     1 < 2
#
# the interacting cagemate is MORE dominant
# than the subject.
#
# -> returns:
#    "higher_than_subject"
#
# This relative-rank classification is later used
# in downstream GLMs examining whether:
# - coherence
# - power
# - granger causality
#
# vary depending on the hierarchy relationship
# between the subject and the interacting cagemate.
# --------------------------------------------------

def relative_rank(row):

    # extract subject UM hierarchy rank
    subj = row["subject_um_rank"]

    # extract partner UM hierarchy rank
    partner = row["partner_um_rank"]

    # return NaN if either hierarchy rank is missing
    if pd.isna(subj) or pd.isna(partner):

        return np.nan

    # lower numerical rank = more dominant

    # partner is more dominant than subject
    if partner < subj:

        return "higher_than_subject"

    # partner is less dominant than subject
    elif partner > subj:

        return "lower_than_subject"

    # partner and subject share the same rank
    else:

        return "equal_rank"

In [ ]:
# Apply the relative-rank classification function
# across all neural-analysis observations in the dataframe.
#
# For each observation, the function compares:
# - the subject mouse's UM hierarchy rank
# - the interacting cagemate's UM hierarchy rank
#
# and assigns a subject-relative hierarchy label:
#
# - "higher_than_subject"
#     -> interacting cagemate is more dominant
#
# - "lower_than_subject"
#     -> interacting cagemate is less dominant
#
# - "equal_rank"
#     -> subject and cagemate share the same rank
#
# The resulting hierarchy relationship is stored in:
#     df["relative_rank"]
#
# This variable is later used in downstream GLMs
# to test whether neural spectral measurements
# (e.g., coherence, power, granger causality)
# differ depending on the hierarchy relationship
# between the subject and the interacting
# repeating/novel cagemate.
#
# axis=1 tells pandas to apply the function
# row-by-row across the dataframe.

df["relative_rank"] = df.apply(
    relative_rank,
    axis=1
)

In [ ]:
import pickle

In [ ]:
with open(
    r"C:\Users\sjs93\Downloads\behavior_dicts_from_frames.pkl",
    "rb"
) as f:

    behavior_dict = pickle.load(f)

In [ ]:
df[
    [
        "partner_mouse",
        "subject_um_rank",
        "partner_um_rank",
        "relative_rank"
    ]
].isna().sum()

In [ ]:
df["relative_rank"].value_counts()

In [ ]:
df[
    [
        "subject",
        "partner",
        "partner_mouse",
        "subject_um_rank",
        "partner_um_rank",
        "relative_rank"
    ]
].drop_duplicates().sort_values(
    ["subject", "partner"]
)

In [ ]:
df["event_length"].describe()

In [ ]:
df["event"].value_counts()

In [ ]:
df.to_csv(
    r"C:\Users\sjs93\OneDrive\Documents\GitHub\diff_fam_social_memory_ephys\other_peoples_sutff\sequioa\Phase1_hab_dis_rank_GLM\lfp_kinematics_habit_dishabit_relative_rank_COMPLETE.csv",
    index=False
)